<a href="https://colab.research.google.com/github/iipingebnaa/Intensive-Big-Data-Analytics-Project/blob/main/01_Data_Ingestion_Cleaning/Data_Ingestion_%26_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Role: Data Engineer

1.   Bernadette Iipinge
2.   226162389



Step 1: Setup and Library Imports

In [1]:
# RetailMart Customer Intelligence Platform
# Deliverable 1: Data Ingestion and Cleaning

!pip install -q pyspark faker

import os
import pandas as pd
import numpy as np
import json
from pathlib import Path

print("RetailMart data ingestion notebook is ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.7 MB/s eta 0:00:00
RetailMart data ingestion notebook is ready.


Step 2: Inspect Available Files and Data Stoarge

In [2]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles currently available:")

for file in os.listdir("/content"):
    print(file)

Current working directory: /content

Files currently available:
.config
.ipynb_checkpoints
drive
sample_data


Step 3: RetailMart Dataset Generation

In [3]:
# ======================================================
# RETAILMART DATASET GENERATOR
# Generates:
#   - retailmart_transactions.csv (1M rows)
#   - customer_feedback.json (250K records)
# ======================================================

# Install required libraries
!pip install -q faker

import pandas as pd
import numpy as np
import json
import uuid
import random
from datetime import datetime, timedelta
from faker import Faker

# ==================== CONFIGURATION ====================
NUM_TRANSACTIONS = 1_000_000
NUM_FEEDBACK = 250_000

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# ==================== REFERENCE DATA ====================

PRODUCT_CATEGORIES = {
    'Electronics': ['Smartphone', 'Laptop', 'Tablet', 'Headphones', 'Smartwatch', 'TV', 'Camera'],
    'Clothing': ['T-Shirt', 'Jeans', 'Jacket', 'Shoes', 'Dress', 'Sweater', 'Shorts'],
    'Home & Kitchen': ['Blender', 'Coffee Maker', 'Cookware Set', 'Toaster', 'Vacuum', 'Dinnerware'],
    'Groceries': ['Milk', 'Bread', 'Eggs', 'Rice', 'Pasta', 'Cereal', 'Snacks'],
    'Beauty': ['Shampoo', 'Moisturizer', 'Perfume', 'Makeup Kit', 'Sunscreen', 'Lipstick'],
    'Sports': ['Yoga Mat', 'Dumbbells', 'Running Shoes', 'Bicycle', 'Tennis Racket', 'Backpack'],
    'Toys': ['Action Figure', 'Board Game', 'Puzzle', 'Doll', 'Building Blocks', 'Remote Car'],
    'Books': ['Novel', 'Textbook', 'Cookbook', 'Biography', 'Comic', 'Self-Help'],
}

STORE_LOCATIONS = ['New York, NY', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'Phoenix, AZ',
                   'Philadelphia, PA', 'San Antonio, TX', 'San Diego, CA', 'Dallas, TX', 'Austin, TX',
                   'San Francisco, CA', 'Boston, MA', 'Seattle, WA', 'Denver, CO', 'Miami, FL',
                   'Atlanta, GA', 'Portland, OR', 'Las Vegas, NV', 'Nashville, TN', 'Minneapolis, MN']

PAYMENT_METHODS = ['Credit Card', 'Debit Card', 'Cash', 'Mobile Pay', 'Gift Card']
GENDERS = ['Male', 'Female', 'Non-Binary']
CHURN_STATUSES = ['Yes', 'No']
FEEDBACK_PLATFORMS = ['Web', 'Mobile App', 'In-Store', 'Email', 'Phone']

POSITIVE_FEEDBACK = [
    "Great products and fast delivery!",
    "Excellent customer service, very helpful.",
    "Love shopping here, always find what I need.",
    "Fantastic experience, will shop again.",
    "Best store ever, highly recommend!",
    "Amazing quality and great prices.",
    "Super fast shipping and great packaging.",
    "Very satisfied with my purchase.",
    "The staff was friendly and knowledgeable.",
    "Perfect shopping experience from start to finish.",
]

NEUTRAL_FEEDBACK = [
    "Products are okay, nothing special.",
    "Standard experience, no complaints.",
    "Decent selection, average prices.",
    "It was fine, nothing remarkable.",
    "Just a regular shopping trip.",
    "Products are as expected.",
    "No issues, but no wow factor either.",
    "Okay experience overall.",
    "Average store, average service.",
    "It works for basic needs.",
]

NEGATIVE_FEEDBACK = [
    "Poor customer service, very disappointed.",
    "Products arrived damaged, terrible packaging.",
    "Long wait times, won't shop here again.",
    "Overpriced and poor quality.",
    "Rude staff and bad experience.",
    "Never received my order, very frustrated.",
    "Products not as described, misleading.",
    "Worst shopping experience ever.",
    "Terrible return policy, very unhelpful.",
    "Would not recommend this store to anyone.",
]

# ==================== TRANSACTION GENERATION ====================

print(f"Generating {NUM_TRANSACTIONS:,} transaction records...")

def generate_transaction(transaction_id):
    """Generate a single transaction record."""

    # Customer demographics
    customer_age = int(np.random.normal(42, 15))
    customer_age = max(18, min(85, customer_age))
    customer_gender = random.choice(GENDERS)

    # Loyalty and visits
    loyalty_years = int(np.random.exponential(3))
    loyalty_years = min(loyalty_years, 10)
    num_visits = int(np.random.poisson(5 + loyalty_years * 2))
    num_visits = max(1, min(num_visits, 100))

    # Product selection
    category = random.choice(list(PRODUCT_CATEGORIES.keys()))
    product_name = random.choice(PRODUCT_CATEGORIES[category])

    # Pricing based on category
    base_prices = {
        'Electronics': 250,
        'Clothing': 45,
        'Home & Kitchen': 85,
        'Groceries': 12,
        'Beauty': 25,
        'Sports': 75,
        'Toys': 35,
        'Books': 18,
    }
    base_price = base_prices.get(category, 50)
    unit_price = round(max(5.0, np.random.normal(base_price, base_price * 0.3)), 2)

    # Quantity
    quantity = int(np.random.exponential(2)) + 1
    quantity = min(quantity, 20)

    # Total amount
    total_amount = round(unit_price * quantity, 2)

    # Store location
    store_location = random.choice(STORE_LOCATIONS)

    # Payment method
    payment_method = random.choice(PAYMENT_METHODS)

    # Transaction date
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2026, 6, 30)
    days_diff = (end_date - start_date).days
    random_days = random.randint(0, days_diff)
    transaction_date = start_date + timedelta(days=random_days,
                                               hours=random.randint(8, 21),
                                               minutes=random.randint(0, 59),
                                               seconds=random.randint(0, 59))

    # Churn probability based on behaviour
    churn_prob = 0.1
    if loyalty_years < 1:
        churn_prob += 0.25
    if num_visits < 3:
        churn_prob += 0.20
    if total_amount < 30:
        churn_prob += 0.10
    if customer_age < 25:
        churn_prob += 0.10
    if payment_method == 'Cash':
        churn_prob += 0.05

    churn = 'Yes' if random.random() < min(churn_prob, 0.85) else 'No'

    return {
        'transaction_id': transaction_id,
        'customer_id': str(uuid.uuid4()),
        'transaction_date': transaction_date.strftime('%Y-%m-%d %H:%M:%S'),
        'product_category': category,
        'product_name': product_name,
        'quantity': quantity,
        'unit_price': unit_price,
        'total_amount': total_amount,
        'store_location': store_location,
        'payment_method': payment_method,
        'customer_age': customer_age,
        'customer_gender': customer_gender,
        'loyalty_years': loyalty_years,
        'num_visits': num_visits,
        'churn': churn,
    }

# Generate transactions in chunks
print("Generating transactions...")
transaction_data = []
chunk_size = 100_000

for i in range(NUM_TRANSACTIONS):
    transaction_data.append(generate_transaction(f"TXN-{i+1:08d}"))

    if (i + 1) % chunk_size == 0:
        print(f"  Generated {i + 1:,} transactions...")

# Create DataFrame
df_transactions = pd.DataFrame(transaction_data)
print(f"✅ Generated {len(df_transactions):,} transaction records")

# ======================================================
# INTRODUCE DATA QUALITY ISSUES (for Unit 1 practical)
# ======================================================

print("\n🔧 Introducing data quality issues...")

# 1. NULL VALUES
print("  - Adding null values...")

# Null in customer_gender (4%)
mask = np.random.random(len(df_transactions)) < 0.04
df_transactions.loc[mask, 'customer_gender'] = np.nan

# Null in customer_age (3%)
mask = np.random.random(len(df_transactions)) < 0.03
df_transactions.loc[mask, 'customer_age'] = np.nan

# Null in payment_method (2%)
mask = np.random.random(len(df_transactions)) < 0.02
df_transactions.loc[mask, 'payment_method'] = np.nan

# Null in total_amount (1%)
mask = np.random.random(len(df_transactions)) < 0.01
df_transactions.loc[mask, 'total_amount'] = np.nan

# Null in customer_id (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
df_transactions.loc[mask, 'customer_id'] = np.nan

# 2. BLANK STRINGS
print("  - Adding blank strings...")

mask = np.random.random(len(df_transactions)) < 0.01
df_transactions.loc[mask, 'customer_gender'] = ''

mask = np.random.random(len(df_transactions)) < 0.005
df_transactions.loc[mask, 'store_location'] = ''

# 3. DUPLICATE transaction_ids (0.5%)
print("  - Adding duplicate transaction_ids...")
dup_count = int(len(df_transactions) * 0.005)
dup_indices = np.random.choice(len(df_transactions), size=dup_count * 2, replace=False)

new_ids = [f"DUP-{i+1:08d}" for i in range(dup_count)]
for i in range(dup_count):
    df_transactions.loc[dup_indices[i], 'transaction_id'] = new_ids[i]
    df_transactions.loc[dup_indices[i + dup_count], 'transaction_id'] = new_ids[i]

# 4. INVALID DATE FORMATS (2%)
print("  - Adding invalid date formats...")
invalid_dates = [
    '2026/01/15 14:30:00',
    '15-01-2026 09:00:00',
    '01/15/2026',
    '2026-01-15',
    'invalid-date',
    '2026-13-45 25:00:00',
    '2026-01-15 14:30',
    '2026-01-15T14:30:00',
]
mask = np.random.random(len(df_transactions)) < 0.02
date_issue_indices = np.where(mask)[0]
for idx in date_issue_indices:
    df_transactions.loc[idx, 'transaction_date'] = random.choice(invalid_dates)

# 5. NEGATIVE VALUES
print("  - Adding negative values...")

# Negative customer_age (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'customer_age'] = -random.randint(1, 20)

# Negative total_amount (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'total_amount'] = -round(random.uniform(5, 50), 2)

# 6. OUT-OF-RANGE VALUES
print("  - Adding out-of-range values...")

# customer_age > 120 (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'customer_age'] = random.randint(121, 150)

# num_visits > 100 (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'num_visits'] = random.randint(101, 500)

# 7. INCONSISTENT CATEGORICAL VALUES
print("  - Adding inconsistent categorical values...")

# Unexpected gender values (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
unexpected_genders = ['M', 'F', 'Other', 'X', 'Prefer not to say']
df_transactions.loc[mask, 'customer_gender'] = np.random.choice(unexpected_genders, size=mask.sum())

# Unexpected payment methods (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
unexpected_payments = ['Bitcoin', 'Cash on Delivery', 'Cheque', 'PayPal', 'Venmo']
df_transactions.loc[mask, 'payment_method'] = np.random.choice(unexpected_payments, size=mask.sum())

# Unexpected churn values (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
inconsistent_churn = ['Y', 'N', 'TRUE', 'FALSE', '1', '0']
df_transactions.loc[mask, 'churn'] = np.random.choice(inconsistent_churn, size=mask.sum())

# ======================================================
# SAVE TRANSACTIONS
# ======================================================

print("\n💾 Saving transactions to CSV...")
output_csv = 'retailmart_transactions.csv'
df_transactions.to_csv(output_csv, index=False)
print(f"✅ Saved: {output_csv}")
print(f"   - Rows: {len(df_transactions):,}")
print(f"   - Columns: {len(df_transactions.columns)}")
print(f"   - File size: {df_transactions.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# ==================== FEEDBACK GENERATION ====================

print(f"\nGenerating {NUM_FEEDBACK:,} customer feedback records...")

def generate_feedback():
    """Generate a single feedback record with nested JSON structure."""

    # Sentiment
    sentiment = round(np.random.uniform(-1, 1), 3)

    # Select feedback text based on sentiment
    if sentiment > 0.3:
        feedback_text = random.choice(POSITIVE_FEEDBACK)
        rating = random.randint(4, 5)
    elif sentiment < -0.3:
        feedback_text = random.choice(NEGATIVE_FEEDBACK)
        rating = random.randint(1, 2)
    else:
        feedback_text = random.choice(NEUTRAL_FEEDBACK)
        rating = 3

    # Customer info
    customer_name = fake.name()
    customer_location = random.choice(STORE_LOCATIONS)

    # Product category mentioned
    product_category = random.choice(list(PRODUCT_CATEGORIES.keys()))

    # Store location
    store_location = random.choice(STORE_LOCATIONS)

    # Timestamp
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2026, 6, 30)
    days_diff = (end_date - start_date).days
    random_days = random.randint(0, days_diff)
    timestamp = start_date + timedelta(days=random_days,
                                        hours=random.randint(8, 21),
                                        minutes=random.randint(0, 59))

    # Platform
    platform = random.choice(FEEDBACK_PLATFORMS)

    return {
        'feedback_id': str(uuid.uuid4()),
        'customer': {
            'id': str(uuid.uuid4()),
            'name': customer_name,
            'location': customer_location,
            'age': int(np.random.normal(42, 15)),
            'loyalty_years': int(np.random.exponential(3)),
        },
        'feedback': {
            'text': feedback_text,
            'rating': rating,
            'sentiment': sentiment,
            'product_category': product_category,
            'store_location': store_location,
        },
        'timestamp': timestamp.isoformat(),
        'platform': platform,
        'device_type': random.choice(['iPhone', 'Android', 'Web Browser', 'Tablet']),
    }

# Generate feedback
print("Generating feedback records...")
feedback_data = []
chunk_size = 50_000

for i in range(NUM_FEEDBACK):
    feedback_data.append(generate_feedback())

    if (i + 1) % chunk_size == 0:
        print(f"  Generated {i + 1:,} feedback records...")

# Save as JSON Lines (one JSON object per line)
print("\n💾 Saving feedback to JSON...")
output_json = 'customer_feedback.json'
with open(output_json, 'w') as f:
    for record in feedback_data:
        f.write(json.dumps(record) + '\n')

print(f"✅ Saved: {output_json}")
print(f"   - Records: {len(feedback_data):,}")

# ======================================================
# SUMMARY
# ======================================================

print("\n" + "=" * 70)
print("✅ RETAILMART DATASETS GENERATED SUCCESSFULLY!")
print("=" * 70)
print(f"📁 {output_csv} ({len(df_transactions):,} rows)")
print(f"📁 {output_json} ({len(feedback_data):,} records)")
print("\n" + "=" * 70)
print("📊 DATA QUALITY ISSUES INTRODUCED:")
print("=" * 70)
print("""
| Issue Type          | Column(s)          | Approximate % |
|---------------------|--------------------| --------------|
| Null values         | customer_gender    | 4%            |
| Null values         | customer_age       | 3%            |
| Null values         | payment_method     | 2%            |
| Null values         | total_amount       | 1%            |
| Null values         | customer_id        | 0.5%          |
| Blank strings       | customer_gender    | 1%            |
| Blank strings       | store_location     | 0.5%          |
| Duplicate IDs       | transaction_id     | 0.5%          |
| Invalid dates       | transaction_date   | 2%            |
| Negative values     | customer_age       | 0.3%          |
| Negative values     | total_amount       | 0.3%          |
| Out-of-range        | customer_age (>120)| 0.3%          |
| Out-of-range        | num_visits (>100)  | 0.3%          |
| Unexpected values   | customer_gender    | 0.5%          |
| Unexpected values   | payment_method     | 0.5%          |
| Inconsistent values | churn              | 0.3%          |
""")

# ======================================================
# DOWNLOAD INSTRUCTIONS
# ======================================================

print("=" * 70)
print("⬇️ TO DOWNLOAD THE DATASETS")
print("=" * 70)
print("""
1. Click the folder icon in the left sidebar of Colab
2. Right-click on 'retailmart_transactions.csv' → Download
3. Right-click on 'customer_feedback.json' → Download
4. Share both files with your group members
""")

# Auto-download (uncomment to use)
# from google.colab import files
# files.download('retailmart_transactions.csv')
# files.download('customer_feedback.json')

print("\n✅ Dataset generation complete!")

Generating 1,000,000 transaction records...
Generating transactions...
  Generated 100,000 transactions...
  Generated 200,000 transactions...
  Generated 300,000 transactions...
  Generated 400,000 transactions...
  Generated 500,000 transactions...
  Generated 600,000 transactions...
  Generated 700,000 transactions...
  Generated 800,000 transactions...
  Generated 900,000 transactions...
  Generated 1,000,000 transactions...
✅ Generated 1,000,000 transaction records

🔧 Introducing data quality issues...
  - Adding null values...
  - Adding blank strings...
  - Adding duplicate transaction_ids...
  - Adding invalid date formats...
  - Adding negative values...
  - Adding out-of-range values...
  - Adding inconsistent categorical values...

💾 Saving transactions to CSV...
✅ Saved: retailmart_transactions.csv
   - Rows: 1,000,000
   - Columns: 15
   - File size: 571.89 MB

Generating 250,000 customer feedback records...
Generating feedback records...
  Generated 50,000 feedback record

Step 4: Dataset Verification.

In [4]:

from pathlib import Path
import json
import pandas as pd

# Check whether both datasets exist
transactions_file = Path("/content/retailmart_transactions.csv")
feedback_file = Path("/content/customer_feedback.json")

for file in [transactions_file, feedback_file]:
    if file.exists():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"Found: {file.name}")
        print(f"File size: {size_mb:.2f} MB\n")
    else:
        print(f"WARNING: {file.name} was not found.")

# Preview the first five transaction records
if transactions_file.exists():
    print("TRANSACTION DATA — FIRST 5 ROWS")
    display(pd.read_csv(transactions_file, nrows=5))

# Preview the first two feedback records
if feedback_file.exists():
    print("\nCUSTOMER FEEDBACK — FIRST 2 RECORDS")
    with open(feedback_file, "r", encoding="utf-8") as f:
        for _ in range(2):
            line = f.readline()
            if line:
                display(json.loads(line))


Found: retailmart_transactions.csv
File size: 139.55 MB

Found: customer_feedback.json
File size: 106.44 MB

TRANSACTION DATA — FIRST 5 ROWS


,transaction_id,customer_id,transaction_date,product_category,product_name,quantity,unit_price,total_amount,store_location,payment_method,customer_age,customer_gender,loyalty_years,num_visits,churn
0,TXN-00000001,16003b4d-1a3a-47e6-a6f9-034d29b08a46,2022-04-03 10:47:06,Clothing,T-Shirt,1,43.13,43.13,"Dallas, TX",Debit Card,49.0,Non-Binary,3,12,No
1,TXN-00000002,a78d77f2-3f2f-4435-8922-2c2a7b67f83a,2021-03-03 09:13:14,Clothing,Dress,1,55.36,55.36,"Denver, CO",Credit Card,65.0,NaN,0,4,No
2,TXN-00000003,11a02b33-f74f-41f1-8e85-af673580f755,2022-03-28 15:37:17,Groceries,Cereal,2,15.42,30.84,"Las Vegas, NV",Mobile Pay,18.0,Male,1,6,No
3,TXN-00000004,ff20f8e0-d4ef-4034-b7a5-3403b31cd211,2022-07-24 10:13:48,Home & Kitchen,Dinnerware,1,87.83,87.83,"Denver, CO",Cash,33.0,Male,2,8,No
4,TXN-00000005,819bde79-261e-4461-8757-c69966605c2d,2024-05-21 12:51:02,Toys,Action Figure,1,34.94,34.94,"Boston, MA",Cash,33.0,Male,7,16,No



CUSTOMER FEEDBACK — FIRST 2 RECORDS


{'feedback_id': '6acd1783-8170-4a6e-8a7f-d942ffb86cf3',
 'customer': {'id': 'ea42e8d9-6cf9-4da5-87a7-c703984d128e',
  'name': 'Allison Hill',
  'location': 'New York, NY',
  'age': 43,
  'loyalty_years': 2},
 'feedback': {'text': 'Products are okay, nothing special.',
  'rating': 3,
  'sentiment': 0.129,
  'product_category': 'Beauty',
  'store_location': 'Houston, TX'},
 'timestamp': '2022-04-27T14:11:00',
 'platform': 'Web',
 'device_type': 'Web Browser'}

{'feedback_id': 'f65f9bf3-23fb-48f5-b23b-dd7f87028ad4',
 'customer': {'id': '0e61176a-a644-46d2-94ed-216a2055a669',
  'name': 'Noah Rhodes',
  'location': 'Atlanta, GA',
  'age': 32,
  'loyalty_years': 9},
 'feedback': {'text': "Long wait times, won't shop here again.",
  'rating': 2,
  'sentiment': -0.841,
  'product_category': 'Books',
  'store_location': 'Atlanta, GA'},
 'timestamp': '2025-02-18T21:47:00',
 'platform': 'Email',
 'device_type': 'Android'}

Step 5: Saving the Raw Datasets to Google Drive

In [5]:

from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount("/content/drive")

# Create a project data folder
project_data_dir = Path(
    "/content/drive/MyDrive/RetailMart_Customer_Intelligence/Data/Raw"
)
project_data_dir.mkdir(parents=True, exist_ok=True)

# Copy the original datasets to Google Drive
raw_files = [
    "/content/retailmart_transactions.csv",
    "/content/customer_feedback.json"
]

for source in raw_files:
    source_path = Path(source)

    if source_path.exists():
        destination = project_data_dir / source_path.name
        shutil.copy2(source_path, destination)
        print(f"Backed up: {source_path.name}")
    else:
        print(f"File not found: {source_path.name}")

print("\nRaw dataset backup completed.")
print("Backup folder:", project_data_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backed up: retailmart_transactions.csv
Backed up: customer_feedback.json

Raw dataset backup completed.
Backup folder: /content/drive/MyDrive/RetailMart_Customer_Intelligence/Data/Raw


Step 6: Transaction Data Profiling

In [6]:

import pandas as pd

# Load the transaction dataset
transactions_path = (
    "/content/drive/MyDrive/RetailMart_Customer_Intelligence/"
    "Data/Raw/retailmart_transactions.csv"
)

df_transactions = pd.read_csv(transactions_path)

# Inspect the dataset
print("Dataset dimensions:")
print(f"Rows: {df_transactions.shape[0]:,}")
print(f"Columns: {df_transactions.shape[1]}")

print("\nColumn names:")
print(df_transactions.columns.tolist())

print("\nDataset information:")
df_transactions.info()

print("\nFirst 5 records:")
display(df_transactions.head())

print("\nDescriptive statistics:")
display(df_transactions.describe(include="all").T)


Dataset dimensions:
Rows: 1,000,000
Columns: 15

Column names:
['transaction_id', 'customer_id', 'transaction_date', 'product_category', 'product_name', 'quantity', 'unit_price', 'total_amount', 'store_location', 'payment_method', 'customer_age', 'customer_gender', 'loyalty_years', 'num_visits', 'churn']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   transaction_id    1000000 non-null  object 
 1   customer_id       994913 non-null   object 
 2   transaction_date  1000000 non-null  object 
 3   product_category  1000000 non-null  object 
 4   product_name      1000000 non-null  object 
 5   quantity          1000000 non-null  int64  
 6   unit_price        1000000 non-null  float64
 7   total_amount      990155 non-null   float64
 8   store_location    995050 non-null   object 
 9   payment_method  

,transaction_id,customer_id,transaction_date,product_category,product_name,quantity,unit_price,total_amount,store_location,payment_method,customer_age,customer_gender,loyalty_years,num_visits,churn
0,TXN-00000001,16003b4d-1a3a-47e6-a6f9-034d29b08a46,2022-04-03 10:47:06,Clothing,T-Shirt,1,43.13,43.13,"Dallas, TX",Debit Card,49.0,Non-Binary,3,12,No
1,TXN-00000002,a78d77f2-3f2f-4435-8922-2c2a7b67f83a,2021-03-03 09:13:14,Clothing,Dress,1,55.36,55.36,"Denver, CO",Credit Card,65.0,NaN,0,4,No
2,TXN-00000003,11a02b33-f74f-41f1-8e85-af673580f755,2022-03-28 15:37:17,Groceries,Cereal,2,15.42,30.84,"Las Vegas, NV",Mobile Pay,18.0,Male,1,6,No
3,TXN-00000004,ff20f8e0-d4ef-4034-b7a5-3403b31cd211,2022-07-24 10:13:48,Home & Kitchen,Dinnerware,1,87.83,87.83,"Denver, CO",Cash,33.0,Male,2,8,No
4,TXN-00000005,819bde79-261e-4461-8757-c69966605c2d,2024-05-21 12:51:02,Toys,Action Figure,1,34.94,34.94,"Boston, MA",Cash,33.0,Male,7,16,No



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,1000000,995000,DUP-00000711,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,994913,994913,e07753db-5270-4cb8-af29-e9f60d3dda9d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_date,1000000,975315,invalid-date,2537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_category,1000000,8,Electronics,125429,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,1000000,51,Novel,21086,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,1000000.0,NaN,NaN,NaN,2.542265,1.977384,1.0,1.0,2.0,3.0,20.0
unit_price,1000000.0,NaN,NaN,NaN,68.218303,78.901866,5.0,19.95,38.57,79.5625,574.33
total_amount,990155.0,NaN,NaN,NaN,172.902448,287.289853,-16.77,35.15,77.84,188.44,7565.4
store_location,995050,20,"Chicago, IL",50058,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_method,979897,10,Mobile Pay,195108,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Step 7: Data Quality Assessment

In [7]:

# 1. Missing values
print("1. MISSING VALUES")
display(
    df_transactions.isnull().sum()
    .to_frame("Missing_Count")
    .query("Missing_Count > 0")
)

# 2. Blank strings
print("\n2. BLANK STRINGS")
for col in ["customer_gender", "store_location"]:
    blank_count = (
        df_transactions[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{col}: {blank_count:,}")

# 3. Duplicate transaction IDs
print("\n3. DUPLICATE TRANSACTION IDs")
print(
    "Rows with duplicated transaction IDs:",
    f"{df_transactions['transaction_id'].duplicated(keep=False).sum():,}"
)
print(
    "Number of duplicated ID groups:",
    f"{df_transactions.loc[df_transactions['transaction_id'].duplicated(keep=False), 'transaction_id'].nunique():,}"
)

# 4. Invalid transaction dates
print("\n4. INVALID TRANSACTION DATES")
parsed_dates = pd.to_datetime(
    df_transactions["transaction_date"],
    errors="coerce",
    format="mixed"
)
print(f"Invalid or unparseable dates: {parsed_dates.isna().sum():,}")

# 5. Negative numeric values
print("\n5. NEGATIVE VALUES")
for col in ["customer_age", "total_amount", "quantity", "unit_price",
            "loyalty_years", "num_visits"]:
    print(f"{col}: {(df_transactions[col] < 0).sum():,}")

# 6. Out-of-range values
print("\n6. OUT-OF-RANGE VALUES")
print(
    "Customer age above 120:",
    f"{(df_transactions['customer_age'] > 120).sum():,}"
)
print(
    "Number of visits above 100:",
    f"{(df_transactions['num_visits'] > 100).sum():,}"
)

# 7. Unexpected categorical values
print("\n7. CATEGORICAL VALUE CHECKS")

expected_genders = {"Male", "Female", "Non-Binary"}
unexpected_genders = (
    df_transactions["customer_gender"].notna()
    & ~df_transactions["customer_gender"].isin(expected_genders)
    & df_transactions["customer_gender"].astype("string").str.strip().ne("")
)
print("Unexpected gender values:")
print(df_transactions.loc[unexpected_genders, "customer_gender"].value_counts())

expected_payments = {
    "Credit Card", "Debit Card", "Cash", "Mobile Pay", "Gift Card"
}
unexpected_payments = (
    df_transactions["payment_method"].notna()
    & ~df_transactions["payment_method"].isin(expected_payments)
)
print("\nUnexpected payment methods:")
print(df_transactions.loc[unexpected_payments, "payment_method"].value_counts())

expected_churn = {"Yes", "No"}
unexpected_churn = (
    df_transactions["churn"].notna()
    & ~df_transactions["churn"].isin(expected_churn)
)
print("\nInconsistent churn values:")
print(df_transactions.loc[unexpected_churn, "churn"].value_counts())

print("\nData quality assessment completed.")


1. MISSING VALUES


,Missing_Count
customer_id,5087
total_amount,9845
store_location,4950
payment_method,20103
customer_age,29756
customer_gender,48956



2. BLANK STRINGS
customer_gender: 0
store_location: 0

3. DUPLICATE TRANSACTION IDs
Rows with duplicated transaction IDs: 10,000
Number of duplicated ID groups: 5,000

4. INVALID TRANSACTION DATES
Invalid or unparseable dates: 4,946

5. NEGATIVE VALUES
customer_age: 3,088
total_amount: 2,970
quantity: 0
unit_price: 0
loyalty_years: 0
num_visits: 0

6. OUT-OF-RANGE VALUES
Customer age above 120: 3,019
Number of visits above 100: 2,960

7. CATEGORICAL VALUE CHECKS
Unexpected gender values:
customer_gender
Other                1048
Prefer not to say    1039
F                    1031
X                    1016
M                    1010
Name: count, dtype: int64

Unexpected payment methods:
payment_method
PayPal              1090
Cheque              1017
Venmo                984
Bitcoin              971
Cash on Delivery     941
Name: count, dtype: int64

Inconsistent churn values:
churn
FALSE    527
N        497
Y        491
1        490
TRUE     469
0        462
Name: count, dtype: int64



Step 8: Cleaning the Transaction Dataset

In [8]:

# Create a separate copy so the original data remains unchanged
df_clean = df_transactions.copy()

# Record the original number of rows
original_rows = len(df_clean)

print("Starting cleaning process...")
print(f"Original rows: {original_rows:,}")

# 1. Standardise blank strings and whitespace
text_columns = [
    "customer_id", "transaction_id", "transaction_date",
    "product_category", "product_name", "store_location",
    "payment_method", "customer_gender", "churn"
]

for col in text_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()
    df_clean[col] = df_clean[col].replace("", pd.NA)

# 2. Standardise known categorical variants
gender_mapping = {
    "M": "Male",
    "F": "Female"
}
df_clean["customer_gender"] = (
    df_clean["customer_gender"].replace(gender_mapping)
)

# Map common churn variants to Yes/No
churn_mapping = {
    "Y": "Yes",
    "N": "No",
    "TRUE": "Yes",
    "FALSE": "No",
    "1": "Yes",
    "0": "No"
}
df_clean["churn"] = df_clean["churn"].replace(churn_mapping)

# Keep unexpected categories visible as missing for consistent analysis
valid_genders = {"Male", "Female", "Non-Binary"}
df_clean.loc[
    ~df_clean["customer_gender"].isin(valid_genders),
    "customer_gender"
] = pd.NA

valid_payments = {
    "Credit Card", "Debit Card", "Cash", "Mobile Pay", "Gift Card"
}
df_clean.loc[
    ~df_clean["payment_method"].isin(valid_payments),
    "payment_method"
] = pd.NA

# 3. Convert data types
df_clean["transaction_date"] = pd.to_datetime(
    df_clean["transaction_date"],
    errors="coerce",
    format="mixed"
)

numeric_columns = [
    "quantity", "unit_price", "total_amount",
    "customer_age", "loyalty_years", "num_visits"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 4. Remove rows without essential identifiers
df_clean = df_clean.dropna(
    subset=["transaction_id", "customer_id"]
)

# 5. Remove duplicate transaction IDs
df_clean = df_clean.drop_duplicates(
    subset=["transaction_id"],
    keep="first"
)

# 6. Remove invalid dates
df_clean = df_clean.dropna(subset=["transaction_date"])

# 7. Remove impossible numeric values
df_clean = df_clean[
    (df_clean["customer_age"].isna() |
     df_clean["customer_age"].between(18, 120))
    & (df_clean["total_amount"].isna() |
       (df_clean["total_amount"] >= 0))
    & (df_clean["quantity"].isna() |
       (df_clean["quantity"] > 0))
    & (df_clean["unit_price"].isna() |
       (df_clean["unit_price"] >= 0))
    & (df_clean["loyalty_years"].isna() |
       (df_clean["loyalty_years"] >= 0))
    & (df_clean["num_visits"].isna() |
       df_clean["num_visits"].between(0, 100))
].copy()

# 8. Impute remaining missing values
# Use median for numeric columns where appropriate
for col in ["customer_age", "total_amount"]:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)

# Use Unknown for missing categorical information
for col in ["customer_gender", "store_location", "payment_method"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

# Fill missing churn with Unknown rather than guessing a label
df_clean["churn"] = df_clean["churn"].fillna("Unknown")

# 9. Final cleaning summary
print("\nCleaning completed.")
print(f"Original rows: {original_rows:,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Rows removed: {original_rows - len(df_clean):,}")
print(f"Rows retained: {len(df_clean) / original_rows * 100:.2f}%")

print("\nRemaining missing values:")
display(
    df_clean.isna().sum()
    .to_frame("Missing_Count")
    .query("Missing_Count > 0")
)

print("\nCleaned data preview:")
display(df_clean.head())


Starting cleaning process...
Original rows: 1,000,000

Cleaning completed.
Original rows: 1,000,000
Cleaned rows: 973,266
Rows removed: 26,734
Rows retained: 97.33%

Remaining missing values:


,Missing_Count



Cleaned data preview:


,transaction_id,customer_id,transaction_date,product_category,product_name,quantity,unit_price,total_amount,store_location,payment_method,customer_age,customer_gender,loyalty_years,num_visits,churn
0,TXN-00000001,16003b4d-1a3a-47e6-a6f9-034d29b08a46,2022-04-03 10:47:06,Clothing,T-Shirt,1,43.13,43.13,"Dallas, TX",Debit Card,49.0,Non-Binary,3,12,No
1,TXN-00000002,a78d77f2-3f2f-4435-8922-2c2a7b67f83a,2021-03-03 09:13:14,Clothing,Dress,1,55.36,55.36,"Denver, CO",Credit Card,65.0,Unknown,0,4,No
2,TXN-00000003,11a02b33-f74f-41f1-8e85-af673580f755,2022-03-28 15:37:17,Groceries,Cereal,2,15.42,30.84,"Las Vegas, NV",Mobile Pay,18.0,Male,1,6,No
3,TXN-00000004,ff20f8e0-d4ef-4034-b7a5-3403b31cd211,2022-07-24 10:13:48,Home & Kitchen,Dinnerware,1,87.83,87.83,"Denver, CO",Cash,33.0,Male,2,8,No
4,TXN-00000005,819bde79-261e-4461-8757-c69966605c2d,2024-05-21 12:51:02,Toys,Action Figure,1,34.94,34.94,"Boston, MA",Cash,33.0,Male,7,16,No


Step 9: Validate the Cleaned Dataset

In [9]:

print("CLEANED DATA VALIDATION")
print("=" * 45)

# 1. Dataset dimensions
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

# 2. Missing values
print("\nMissing values remaining:")
print(df_clean.isna().sum().sum())

# 3. Duplicate transaction IDs
print("\nDuplicate transaction IDs:")
print(df_clean["transaction_id"].duplicated().sum())

# 4. Duplicate customer IDs
# Customer IDs are expected to be unique in this synthetic generator.
print("\nDuplicate customer IDs:")
print(df_clean["customer_id"].duplicated().sum())

# 5. Invalid transaction dates
print("\nInvalid transaction dates:")
print(df_clean["transaction_date"].isna().sum())

# 6. Numeric range checks
print("\nInvalid numeric values:")
print("Age below 18 or above 120:",
      ((df_clean["customer_age"] < 18) |
       (df_clean["customer_age"] > 120)).sum())
print("Negative transaction amounts:",
      (df_clean["total_amount"] < 0).sum())
print("Non-positive quantities:",
      (df_clean["quantity"] <= 0).sum())
print("Visits outside 0–100:",
      ((df_clean["num_visits"] < 0) |
       (df_clean["num_visits"] > 100)).sum())

# 7. Categorical checks
print("\nUnexpected gender values:")
print(
    (~df_clean["customer_gender"].isin(
        ["Male", "Female", "Non-Binary", "Unknown"]
    )).sum()
)

print("\nUnexpected payment methods:")
print(
    (~df_clean["payment_method"].isin(
        ["Credit Card", "Debit Card", "Cash",
         "Mobile Pay", "Gift Card", "Unknown"]
    )).sum()
)

print("\nUnexpected churn values:")
print(
    (~df_clean["churn"].isin(["Yes", "No", "Unknown"])).sum()
)

print("\nValidation checks completed.")


CLEANED DATA VALIDATION
Rows: 973,266
Columns: 15

Missing values remaining:
0

Duplicate transaction IDs:
0

Duplicate customer IDs:
0

Invalid transaction dates:
0

Invalid numeric values:
Age below 18 or above 120: 0
Negative transaction amounts: 0
Non-positive quantities: 0
Visits outside 0–100: 0

Unexpected gender values:
0

Unexpected payment methods:
0

Unexpected churn values:
0

Validation checks completed.


Step 10: Downloading the Cleaned Dataset to Your Computer

In [10]:
from pathlib import Path

# Create a local output folder in Colab
output_dir = Path("/content/RetailMart_Cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

# Define output file paths
parquet_path = output_dir / "retailmart_transactions_cleaned.parquet"
csv_path = output_dir / "retailmart_transactions_cleaned.csv"

# Save the cleaned dataset
df_clean.to_parquet(
    parquet_path,
    index=False,
    compression="snappy"
)

df_clean.to_csv(csv_path, index=False)

# Verify that both files contain data
for file_path in [parquet_path, csv_path]:
    if file_path.exists() and file_path.stat().st_size > 0:
        print(f"{file_path.name}: {file_path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"ERROR: {file_path.name} was not saved correctly.")

retailmart_transactions_cleaned.parquet: 54.85 MB
retailmart_transactions_cleaned.csv: 136.68 MB


Step 11. Downloading the Files

In [11]:
from google.colab import files

files.download(
    "/content/RetailMart_Cleaned/retailmart_transactions_cleaned.parquet"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import files

files.download(
    "/content/RetailMart_Cleaned/retailmart_transactions_cleaned.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>